<a href="https://colab.research.google.com/github/GifariMadia/data-science-portfolio/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import pandas as pd
import kagglehub
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.decomposition import LatentDirichletAllocation

def preprocess_indonesian_text(text: str) -> str:
    """Pembersihan teks dasar."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_and_prep_kaggle_data() -> pd.DataFrame:
    """
    Mengunduh dataset langsung dari Kaggle dan memproses target sentimen.
    """
    print("Mengunduh dataset dari Kaggle...")
    # Menggunakan metode yang Anda temukan
    folder_path = kagglehub.dataset_download("itanium/livin-by-mandiri-app-reviews")
    file_path = os.path.join(folder_path, "reviews.csv")

    print(f"Dataset ditemukan di: {file_path}")
    df = pd.read_csv(file_path)

    # KEPUTUSAN: Hapus baris tanpa ulasan teks
    df = df.dropna(subset=['review'])

    # KEPUTUSAN: Konversi rating menjadi target biner
    # Mengabaikan bintang 3 untuk memisahkan polaritas dengan tegas
    df = df[df['rating'] != 3].copy()
    df['sentiment'] = df['rating'].apply(lambda x: 1 if x > 3 else 0)
    df = df.rename(columns={'review': 'text'})

    # KEPUTUSAN: Karena dataset memiliki ratusan ribu baris, kita ambil sampel
    # 20.000 baris (stratified) agar iterasi training di memori lokal tetap cepat.
    # Di level production, seluruh data akan dilatih di cloud.
    if len(df) > 20000:
        df = df.groupby('sentiment', group_keys=False).apply(lambda x: x.sample(10000, random_state=42))

    return df.reset_index(drop=True)

def train_sentiment_baseline(df: pd.DataFrame):
    """Melatih SVM untuk klasifikasi sentimen biner."""
    print("Membersihkan teks...")
    df['clean_text'] = df['text'].apply(preprocess_indonesian_text)

    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_text'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
    )

    vectorizer = TfidfVectorizer(max_features=2000)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    model = LinearSVC(random_state=42, dual=False)
    model.fit(X_train_tfidf, y_train)

    preds = model.predict(X_test_tfidf)
    print("\n--- Laporan Evaluasi Sentimen (SVM) ---")
    print(classification_report(y_test, preds, target_names=['Negatif', 'Positif']))

    return df

def extract_negative_topics(df: pd.DataFrame):
    """Menggunakan LDA untuk mencari 3 topik utama keluhan."""
    neg_reviews = df[df['sentiment'] == 0]['clean_text']

    tf_vectorizer = CountVectorizer(max_df=0.9, min_df=5, max_features=1000)
    tf_matrix = tf_vectorizer.fit_transform(neg_reviews)

    lda = LatentDirichletAllocation(n_components=3, random_state=42, max_iter=15)
    lda.fit(tf_matrix)

    print("\n--- Tema Keluhan Utama (Berdasarkan Bintang 1 & 2) ---")
    feature_names = tf_vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(lda.components_):
        top_words_idx = topic.argsort()[:-6:-1]
        top_words = [feature_names[i] for i in top_words_idx]
        print(f"Topik {topic_idx + 1}: {', '.join(top_words)}")

# ==========================================
# EKSEKUSI PIPELINE
# ==========================================
df_mandiri = load_and_prep_kaggle_data()
df_clean = train_sentiment_baseline(df_mandiri)
extract_negative_topics(df_clean)

Mengunduh dataset dari Kaggle...


100%|██████████| 4.02M/4.02M [00:00<00:00, 125MB/s]

Extracting files...
Dataset ditemukan di: /root/.cache/kagglehub/datasets/itanium/livin-by-mandiri-app-reviews/versions/2/reviews.csv



/tmp/ipykernel_419/1364275379.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('sentiment', group_keys=False).apply(lambda x: x.sample(10000, random_state=42))


Membersihkan teks...

--- Laporan Evaluasi Sentimen (SVM) ---
              precision    recall  f1-score   support

     Negatif       0.90      0.91      0.91      2000
     Positif       0.91      0.90      0.91      2000

    accuracy                           0.91      4000
   macro avg       0.91      0.91      0.91      4000
weighted avg       0.91      0.91      0.91      4000


--- Tema Keluhan Utama (Berdasarkan Bintang 1 & 2) ---
Topik 1: saya, tidak, bisa, di, ada
Topik 2: yg, biru, livin, aplikasi, sering
Topik 3: bisa, update, di, terus, gak
